# LP Feature Overview

Notebook d'exploration visuelle de toutes les familles de features déjà matérialisées dans le warehouse.

Couverture:
- TVL current / block snapshot / daily / distribution
- volume et fees
- fee APR
- price analytics
- fee-per-100 width curves
- LP macro features
- intraday price/volume
- tick occupancy
- open positions

Objectif: vérifier rapidement si les features racontent une histoire cohérente avant dashboarding dans Metabase.

## Prérequis

- `clickhouse_local` doit tourner sur `localhost:8124`
- les marts doivent être build en local
- si tu veux figer le snapshot block-level sur un block précis:

```bash
cd /home/gharbi/defind-analytics/dbt/defind_analytics
../../.venv/bin/dbt build --target local --select mart_tvl_block_snapshot --vars '{"target_block": 24820333}'
```

In [1]:
import os
from textwrap import dedent

import clickhouse_connect

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'pandas is required. Run: %pip install pandas clickhouse-connect plotly jupyterlab'
    ) from exc

try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ModuleNotFoundError:
    PLOTLY_AVAILABLE = False
    print('Plotly not available. Run: %pip install plotly')

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 200)

In [2]:
CLICKHOUSE_HOST = os.getenv('CLICKHOUSE_HOST', 'localhost')
CLICKHOUSE_PORT = int(os.getenv('CLICKHOUSE_PORT', '8124'))
CLICKHOUSE_USER = os.getenv('CLICKHOUSE_USER', 'default')
CLICKHOUSE_PASSWORD = os.getenv('CLICKHOUSE_PASSWORD', '1234')
CLICKHOUSE_DATABASE = os.getenv('CLICKHOUSE_DATABASE', 'analytics')

client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

print(f'Connected to ClickHouse database={CLICKHOUSE_DATABASE} on {CLICKHOUSE_HOST}:{CLICKHOUSE_PORT}')

Connected to ClickHouse database=analytics on localhost:8124


In [3]:
TIME_COLUMNS = ['date', 'hour', 'snapshot_block_time', 'block_time', 'deployment_ts']

def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    for column in TIME_COLUMNS:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column])
    return df

def query_df(sql: str, parameters: dict | None = None) -> pd.DataFrame:
    df = client.query_df(sql, parameters=parameters or {})
    df = normalize_df(df)
    print(f'{len(df):,} rows')
    return df

def plot_guard():
    if not PLOTLY_AVAILABLE:
        print('Plotly unavailable. Install with: %pip install plotly')
        return False
    return True

## 1. Pool and warehouse coverage

In [4]:
available_pools = query_df(dedent(
    '''
    SELECT pool
    FROM mart_tvl_current
    ORDER BY pool
    '''
))
available_pools

1 rows


,pool
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640


In [5]:
POOL = os.getenv('POOL_ADDRESS', available_pools.iloc[0]['pool'] if not available_pools.empty else None)
assert POOL, 'No pool found in mart_tvl_current.'
print(f'Using pool: {POOL}')

Using pool: 0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640


In [6]:
mart_inventory = query_df(dedent(
    '''
    SELECT
        model,
        rows,
        min_time,
        max_time
    FROM (
        SELECT 'mart_tvl_daily' AS model, count() AS rows, min(toDateTime(date)) AS min_time, max(toDateTime(date)) AS max_time FROM mart_tvl_daily WHERE pool = %(pool)s
        UNION ALL
        SELECT 'mart_volume_daily' AS model, count() AS rows, min(toDateTime(date)) AS min_time, max(toDateTime(date)) AS max_time FROM mart_volume_daily WHERE pool = %(pool)s
        UNION ALL
        SELECT 'mart_fee_yield_daily' AS model, count() AS rows, min(toDateTime(date)) AS min_time, max(toDateTime(date)) AS max_time FROM mart_fee_yield_daily WHERE pool = %(pool)s
        UNION ALL
        SELECT 'mart_price_analytics_daily' AS model, count() AS rows, min(toDateTime(date)) AS min_time, max(toDateTime(date)) AS max_time FROM mart_price_analytics_daily WHERE pool = %(pool)s
        UNION ALL
        SELECT 'int_fee_per_100_usd' AS model, count() AS rows, min(toDateTime(date)) AS min_time, max(toDateTime(date)) AS max_time FROM int_fee_per_100_usd WHERE pool = %(pool)s
        UNION ALL
        SELECT 'int_lp_macro_daily' AS model, count() AS rows, min(toDateTime(date)) AS min_time, max(toDateTime(date)) AS max_time FROM int_lp_macro_daily WHERE pool = %(pool)s
        UNION ALL
        SELECT 'mart_price_intraday' AS model, count() AS rows, min(hour) AS min_time, max(hour) AS max_time FROM mart_price_intraday WHERE pool = %(pool)s
        UNION ALL
        SELECT 'mart_tick_daily' AS model, count() AS rows, min(toDateTime(date)) AS min_time, max(toDateTime(date)) AS max_time FROM mart_tick_daily WHERE pool = %(pool)s
        UNION ALL
        SELECT 'mart_open_positions' AS model, count() AS rows, NULL AS min_time, NULL AS max_time FROM mart_open_positions WHERE pool = %(pool)s
        UNION ALL
        SELECT 'mart_tvl_distribution' AS model, count() AS rows, NULL AS min_time, NULL AS max_time FROM mart_tvl_distribution WHERE pool = %(pool)s
        UNION ALL
        SELECT 'mart_tvl_current' AS model, count() AS rows, NULL AS min_time, NULL AS max_time FROM mart_tvl_current WHERE pool = %(pool)s
        UNION ALL
        SELECT 'mart_tvl_block_snapshot' AS model, count() AS rows, min(snapshot_block_time) AS min_time, max(snapshot_block_time) AS max_time FROM mart_tvl_block_snapshot WHERE pool = %(pool)s
    )
    ORDER BY model
    '''
), {'pool': POOL})
mart_inventory

12 rows


,model,rows,min_time,max_time
0,mart_fee_per_100_usd,10788,2021-05-05 00:00:00,2026-04-06 00:00:00
1,mart_fee_yield_daily,1798,2021-05-05 00:00:00,2026-04-06 00:00:00
2,mart_lp_macro_daily,1798,2021-05-05 00:00:00,2026-04-06 00:00:00
3,mart_open_positions,3145,NaT,NaT
4,mart_price_analytics_daily,1798,2021-05-05 00:00:00,2026-04-06 00:00:00
5,mart_price_intraday,43118,2021-05-05 22:00:00,2026-04-06 11:00:00
6,mart_tick_daily,805633,2021-05-05 00:00:00,2026-04-06 00:00:00
7,mart_tvl_block_snapshot,1,2026-04-06 11:28:11,2026-04-06 11:28:11
8,mart_tvl_current,1,NaT,NaT
9,mart_tvl_daily,1798,2021-05-05 00:00:00,2026-04-06 00:00:00


## 2. TVL features

In [7]:
tvl_snapshot = query_df(dedent(
    '''
    SELECT
        c.pool,
        c.current_tick,
        c.price_token1_in_token0,
        c.total_tvl_token0,
        c.active_tvl_token0,
        c.out_of_range_tvl_token0,
        c.current_bin_tick_lower,
        c.current_bin_tick_upper,
        c.current_bin_liquidity,
        c.current_bin_token0,
        c.current_bin_token1,
        c.current_bin_token1_in_token0,
        c.current_bin_tvl_token0,
        b.target_block,
        b.snapshot_log_index,
        b.snapshot_block_time,
        b.total_tvl_token0 AS block_total_tvl_token0,
        b.active_tvl_token0 AS block_active_tvl_token0,
        b.out_of_range_tvl_token0 AS block_out_of_range_tvl_token0
    FROM mart_tvl_current c
    LEFT JOIN mart_tvl_block_snapshot b ON c.pool = b.pool
    WHERE c.pool = %(pool)s
    '''
), {'pool': POOL})
tvl_snapshot

1 rows


,pool,current_tick,price_token1_in_token0,total_tvl_token0,active_tvl_token0,out_of_range_tvl_token0,current_bin_tick_lower,current_bin_tick_upper,current_bin_liquidity,current_bin_token0,current_bin_token1,current_bin_token1_in_token0,current_bin_tvl_token0,target_block,snapshot_log_index,snapshot_block_time,block_total_tvl_token0,block_active_tvl_token0,block_out_of_range_tvl_token0
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,199587,2150.092004,9.362351e+07,7.460444e+07,1.901907e+07,199580,199590,1.451949e+19,83112.776728,117.876524,253445.372693,336558.14942,24820333,80,2026-04-06 11:28:11,9.362351e+07,7.460444e+07,1.901907e+07


In [8]:
tvl_daily = query_df(dedent(
    '''
    SELECT
        d.date,
        d.tvl_token0_equiv,
        p.close_price,
        p.realized_vol_7d,
        p.realized_vol_30d,
        p.drawdown
    FROM mart_tvl_daily d
    LEFT JOIN mart_price_analytics_daily p
        ON d.pool = p.pool AND d.date = p.date
    WHERE d.pool = %(pool)s
    ORDER BY d.date
    '''
), {'pool': POOL})
tvl_daily.tail(10)

1,798 rows


,date,tvl_token0_equiv,close_price,realized_vol_7d,realized_vol_30d,drawdown
1788,2026-03-28,8.754172e+07,1993.846431,0.556766,0.635767,-0.587389
1789,2026-03-29,8.765537e+07,1983.399845,0.552562,0.612137,-0.589550
1790,2026-03-30,8.852209e+07,2024.662060,0.441951,0.613180,-0.581012
1791,2026-03-31,9.064371e+07,2105.216375,0.545516,0.624541,-0.564341
1792,2026-04-01,9.112170e+07,2139.518675,0.558981,0.608865,-0.557243
1793,2026-04-02,9.127022e+07,2056.720249,0.507319,0.619862,-0.574377
1794,2026-04-03,9.172124e+07,2053.374258,0.437049,0.568163,-0.575070
1795,2026-04-04,9.187064e+07,2064.719705,0.436346,0.561337,-0.572722
1796,2026-04-05,9.265439e+07,2109.333463,0.440054,0.541138,-0.563489
1797,2026-04-06,9.362351e+07,2150.092004,0.438128,0.543693,-0.555055


In [9]:
if plot_guard() and not tvl_daily.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=tvl_daily['date'], y=tvl_daily['tvl_token0_equiv'], mode='lines', name='Total TVL'))
    fig.add_trace(go.Scatter(x=tvl_daily['date'], y=tvl_daily['close_price'], mode='lines', name='Price', yaxis='y2', opacity=0.6))
    fig.update_layout(
        title=f'TVL and price over time - {POOL}',
        yaxis=dict(title='TVL (token0 equiv)'),
        yaxis2=dict(title='Price', overlaying='y', side='right'),
        legend=dict(orientation='h'),
    )
    fig.show()

In [10]:
tvl_distribution = query_df(dedent(
    '''
    SELECT
        tick_lower,
        tick_upper,
        tick_mid,
        is_active_bin,
        liquidity,
        amount0,
        amount1,
        tvl_token0,
        pct_liquidity,
        price_lower,
        price_upper
    FROM mart_tvl_distribution
    WHERE pool = %(pool)s
    ORDER BY tick_lower
    '''
), {'pool': POOL})
tvl_distribution.head(20)

1,537 rows


,tick_lower,tick_upper,tick_mid,is_active_bin,liquidity,amount0,amount1,tvl_token0,pct_liquidity,price_lower,price_upper
0,-887270,-887220,-887245.0,0,1.798268e+16,0.0,2.440370e-24,5.247021e-21,0.000452,3.401887e+50,3.384921e+50
1,-887220,-92110,-489665.0,0,1.798345e+16,0.0,1.798166e-04,3.866222e-01,0.000452,3.384921e+50,1.000199e+16
2,-92110,-76280,-84195.0,0,1.798385e+16,0.0,2.169745e-04,4.665151e-01,0.000452,1.000199e+16,2.054152e+15
3,-76280,-75980,-76130.0,0,2.966900e+19,0.0,9.892754e-03,2.127033e+01,0.746065,2.054152e+15,1.993446e+15
4,-75980,0,-37990.0,0,1.798385e+16,0.0,1.758105e-02,3.780089e+01,0.000452,1.993446e+15,1.000000e+12
5,0,100,50.0,0,1.799742e+16,0.0,9.020791e-05,1.939553e-01,0.000453,1.000000e+12,9.900503e+11
6,100,110,105.0,0,1.799742e+16,0.0,9.045621e-06,1.944892e-02,0.000453,9.900503e+11,9.890608e+11
7,110,100000,50055.0,0,1.799742e+16,0.0,2.652289e+00,5.702666e+03,0.000453,9.890608e+11,4.542263e+07
8,100000,108340,104170.0,0,1.798385e+16,0.0,1.380538e+00,2.968284e+03,0.000452,4.542263e+07,1.972826e+07
9,108340,108360,108350.0,0,1.798385e+16,0.0,4.050733e-03,8.709450e+00,0.000452,1.972826e+07,1.968885e+07


In [11]:
if plot_guard() and not tvl_distribution.empty:
    dist_plot = tvl_distribution.copy()
    fig = px.bar(
        dist_plot,
        x='tick_mid',
        y='tvl_token0',
        color=dist_plot['is_active_bin'].astype(str),
        title=f'Current TVL distribution by tick bin - {POOL}',
        labels={'tick_mid': 'Tick mid', 'tvl_token0': 'TVL (token0 equiv)', 'color': 'is_active_bin'},
    )
    fig.update_layout(bargap=0.02)
    fig.show()

## 3. Volume, fees and APR

In [12]:
volume_fee_daily = query_df(dedent(
    '''
    SELECT
        v.date,
        v.volume_token0_equiv,
        v.fees_token0_equiv,
        v.swap_count,
        f.apr,
        f.tvl_active_token0,
        f.tvl_outofrange_token0
    FROM mart_volume_daily v
    LEFT JOIN mart_fee_yield_daily f
        ON v.pool = f.pool AND v.date = f.date
    WHERE v.pool = %(pool)s
    ORDER BY v.date
    '''
), {'pool': POOL})
volume_fee_daily.tail(10)

1,798 rows


,date,volume_token0_equiv,fees_token0_equiv,swap_count,apr,tvl_active_token0,tvl_outofrange_token0
1788,2026-03-28,8.913167e+07,44565.837138,3620,0.216346,7.518743e+07,1.235428e+07
1789,2026-03-29,1.350284e+08,67514.179688,4207,0.328486,7.501890e+07,1.263647e+07
1790,2026-03-30,2.206654e+08,110332.688755,6242,0.495394,8.129173e+07,7.230359e+06
1791,2026-03-31,2.611193e+08,130559.664071,6386,0.661620,7.202661e+07,1.861709e+07
1792,2026-04-01,3.302779e+08,165138.940637,6563,0.835247,7.216515e+07,1.895655e+07
1793,2026-04-02,2.244882e+08,112244.092442,5386,0.559925,7.316892e+07,1.810130e+07
1794,2026-04-03,9.309868e+07,46549.342765,3061,0.230733,7.363709e+07,1.808415e+07
1795,2026-04-04,3.646022e+07,18230.108858,1791,0.090348,7.364842e+07,1.822222e+07
1796,2026-04-05,9.428245e+07,47141.227895,3354,0.232482,7.401253e+07,1.864187e+07
1797,2026-04-06,8.244784e+07,41223.922986,2248,0.201687,7.460444e+07,1.901907e+07


In [13]:
if plot_guard() and not volume_fee_daily.empty:
    fig = go.Figure()
    fig.add_trace(go.Bar(x=volume_fee_daily['date'], y=volume_fee_daily['volume_token0_equiv'], name='Volume'))
    fig.add_trace(go.Bar(x=volume_fee_daily['date'], y=volume_fee_daily['fees_token0_equiv'], name='Fees'))
    fig.add_trace(go.Scatter(x=volume_fee_daily['date'], y=volume_fee_daily['apr'], mode='lines', name='APR', yaxis='y2'))
    fig.update_layout(
        title=f'Daily volume, fees, and APR - {POOL}',
        barmode='overlay',
        yaxis=dict(title='Token0 equiv'),
        yaxis2=dict(title='APR', overlaying='y', side='right'),
        legend=dict(orientation='h'),
    )
    fig.show()

## 4. Price analytics features

In [14]:
price_analytics = query_df(dedent(
    '''
    SELECT
        date,
        close_tick,
        close_price,
        has_swap,
        log_return,
        realized_vol_7d,
        realized_vol_30d,
        rolling_high,
        drawdown
    FROM mart_price_analytics_daily
    WHERE pool = %(pool)s
    ORDER BY date
    '''
), {'pool': POOL})
price_analytics.tail(10)

1,798 rows


,date,close_tick,close_price,has_swap,log_return,realized_vol_7d,realized_vol_30d,rolling_high,drawdown
1788,2026-03-28,200342,1993.846431,1,0.001147,0.556766,0.635767,4832.262009,-0.587389
1789,2026-03-29,200394,1983.399845,1,-0.005253,0.552562,0.612137,4832.262009,-0.589550
1790,2026-03-30,200188,2024.662060,1,0.020590,0.441951,0.613180,4832.262009,-0.581012
1791,2026-03-31,199798,2105.216375,1,0.039015,0.545516,0.624541,4832.262009,-0.564341
1792,2026-04-01,199636,2139.518675,1,0.016163,0.558981,0.608865,4832.262009,-0.557243
1793,2026-04-02,200031,2056.720249,1,-0.039468,0.507319,0.619862,4832.262009,-0.574377
1794,2026-04-03,200047,2053.374258,1,-0.001628,0.437049,0.568163,4832.262009,-0.575070
1795,2026-04-04,199992,2064.719705,1,0.005510,0.436346,0.561337,4832.262009,-0.572722
1796,2026-04-05,199778,2109.333463,1,0.021378,0.440054,0.541138,4832.262009,-0.563489
1797,2026-04-06,199587,2150.092004,1,0.019139,0.438128,0.543693,4832.262009,-0.555055


In [15]:
if plot_guard() and not price_analytics.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=price_analytics['date'], y=price_analytics['close_price'], mode='lines', name='Close price'))
    fig.add_trace(go.Scatter(x=price_analytics['date'], y=price_analytics['rolling_high'], mode='lines', name='Rolling high', opacity=0.6))
    fig.add_trace(go.Scatter(x=price_analytics['date'], y=price_analytics['drawdown'], mode='lines', name='Drawdown', yaxis='y2'))
    fig.update_layout(
        title=f'Price, rolling high, and drawdown - {POOL}',
        yaxis=dict(title='Price'),
        yaxis2=dict(title='Drawdown', overlaying='y', side='right'),
        legend=dict(orientation='h'),
    )
    fig.show()

In [16]:
if plot_guard() and not price_analytics.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=price_analytics['date'], y=price_analytics['realized_vol_7d'], mode='lines', name='Realized vol 7d'))
    fig.add_trace(go.Scatter(x=price_analytics['date'], y=price_analytics['realized_vol_30d'], mode='lines', name='Realized vol 30d'))
    fig.update_layout(title=f'Realized volatility - {POOL}', legend=dict(orientation='h'))
    fig.show()

## 5. Fee-per-100 width curves

In [17]:
fee_curve_latest = query_df(dedent(
    '''
    SELECT
        date,
        range_width_ticks,
        fee_per_100_token0_daily,
        fee_per_100_token0_7d,
        fee_per_100_token0_30d,
        fee_per_100_token0_90d,
        fee_bps_7d,
        fee_bps_30d,
        fee_bps_90d
    FROM int_fee_per_100_usd
    WHERE pool = %(pool)s
      AND date = (SELECT max(date) FROM int_fee_per_100_usd WHERE pool = %(pool)s)
    ORDER BY range_width_ticks
    '''
), {'pool': POOL})
fee_curve_latest

6 rows


,date,range_width_ticks,fee_per_100_token0_daily,fee_per_100_token0_7d,fee_per_100_token0_30d,fee_per_100_token0_90d,fee_bps_7d,fee_bps_30d,fee_bps_90d
0,2026-04-06,50,2.042667,1575.840051,2270.964808,4389.644021,157584.005070,227096.480828,438964.402084
1,2026-04-06,100,1.114876,860.086046,1253.641464,2439.208233,86008.604597,125364.146367,243920.823257
2,2026-04-06,200,0.584712,451.083502,651.957639,1267.075939,45108.350183,65195.763942,126707.593907
3,2026-04-06,500,0.241667,186.436786,268.088187,520.668465,18643.678553,26808.818741,52066.846517
4,2026-04-06,1000,0.122792,94.729628,135.984877,264.042351,9472.962833,13598.487715,26404.235146
5,2026-04-06,2000,0.062473,48.195369,69.125614,134.205878,4819.536947,6912.561401,13420.587789


In [18]:
if plot_guard() and not fee_curve_latest.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=fee_curve_latest['range_width_ticks'], y=fee_curve_latest['fee_bps_7d'], mode='lines+markers', name='7d bps'))
    fig.add_trace(go.Scatter(x=fee_curve_latest['range_width_ticks'], y=fee_curve_latest['fee_bps_30d'], mode='lines+markers', name='30d bps'))
    fig.add_trace(go.Scatter(x=fee_curve_latest['range_width_ticks'], y=fee_curve_latest['fee_bps_90d'], mode='lines+markers', name='90d bps'))
    fig.update_layout(title=f'Latest fee curve by width - {POOL}', xaxis_title='Range width (ticks)', yaxis_title='Fee bps')
    fig.show()

In [19]:
fee_curve_history = query_df(dedent(
    '''
    SELECT
        date,
        range_width_ticks,
        fee_bps_7d
    FROM int_fee_per_100_usd
    WHERE pool = %(pool)s
      AND range_width_ticks IN (100, 500, 1000, 2000)
    ORDER BY date, range_width_ticks
    '''
), {'pool': POOL})
fee_curve_history.tail(20)

7,192 rows


,date,range_width_ticks,fee_bps_7d
7172,2026-04-02,100,109602.685901
7173,2026-04-02,500,23758.066844
7174,2026-04-02,1000,12071.614384
7175,2026-04-02,2000,6141.646959
7176,2026-04-03,100,100923.982279
7177,2026-04-03,500,21876.817825
7178,2026-04-03,1000,11115.740165
7179,2026-04-03,2000,5655.328954
7180,2026-04-04,100,97126.364538
7181,2026-04-04,500,21053.626986


In [20]:
if plot_guard() and not fee_curve_history.empty:
    fig = px.line(
        fee_curve_history,
        x='date',
        y='fee_bps_7d',
        color='range_width_ticks',
        title=f'7d fee curve history by width - {POOL}',
        labels={'fee_bps_7d': 'Fee bps 7d', 'range_width_ticks': 'Width'},
    )
    fig.show()

## 6. LP macro features

In [21]:
lp_macro = query_df(dedent(
    '''
    SELECT
        date,
        tvl_token0_equiv,
        active_tvl_token0,
        active_range_count,
        fees_token0_equiv,
        volume_token0_equiv,
        swap_count,
        capital_utilization,
        volume_tvl_ratio,
        active_liquidity_depth,
        realized_vol_7d,
        apr_daily,
        apr_7d,
        apr_30d,
        apr7_vol_ratio,
        coupon_vol_ratio_500,
        mint_count,
        burn_count,
        mint_token0,
        burn_token0,
        net_lp_flow_token0,
        lp_turnover,
        pct_burns_inrange,
        open_range_count,
        range_width_median_ticks,
        range_width_p25_ticks,
        range_width_p75_ticks,
        lp_concentration_top10_pct,
        range_overlap_density
    FROM int_lp_macro_daily
    WHERE pool = %(pool)s
    ORDER BY date
    '''
), {'pool': POOL})
lp_macro.tail(10)

1,798 rows


,date,tvl_token0_equiv,active_tvl_token0,active_range_count,fees_token0_equiv,volume_token0_equiv,swap_count,capital_utilization,volume_tvl_ratio,active_liquidity_depth,realized_vol_7d,apr_daily,apr_7d,apr_30d,apr7_vol_ratio,coupon_vol_ratio_500,mint_count,burn_count,mint_token0,burn_token0,net_lp_flow_token0,lp_turnover,pct_burns_inrange,open_range_count,range_width_median_ticks,range_width_p25_ticks,range_width_p75_ticks,lp_concentration_top10_pct,range_overlap_density
1788,2026-03-28,8.754172e+07,7.518743e+07,1426,44565.837138,8.913167e+07,3620,0.858875,1.018162,0.843555,0.556766,0.216346,0.513316,0.603612,0.921960,438.108264,161,138,3.531468e+07,3.468490e+07,6.297780e+05,0.396210,0.681159,3071,410.0,300.0,1480.0,0.727288,1434.810
1789,2026-03-29,8.765537e+07,7.501890e+07,1444,67514.179688,1.350284e+08,4207,0.855839,1.540446,0.555579,0.552562,0.328486,0.480361,0.589606,0.869333,409.678735,123,120,3.677671e+07,3.640292e+07,3.737892e+05,0.415296,0.591667,3096,400.0,300.0,1480.0,0.724419,1443.710
1790,2026-03-30,8.852209e+07,8.129173e+07,1506,110332.688755,2.206654e+08,6242,0.918321,2.492772,0.368394,0.441951,0.495394,0.416168,0.583429,0.941662,435.121347,170,196,5.225468e+07,5.236195e+07,-1.072696e+05,0.591513,0.719388,3103,410.0,300.0,1480.0,0.724328,1505.920
1791,2026-03-31,9.064371e+07,7.202661e+07,1531,130559.664071,2.611193e+08,6386,0.794612,2.880722,0.275838,0.545516,0.661620,0.439589,0.572187,0.805822,371.812119,169,166,1.670108e+07,1.610708e+07,5.939978e+05,0.177697,0.596386,3123,410.0,300.0,1480.0,0.717143,1527.130
1792,2026-04-01,9.112170e+07,7.216515e+07,1506,165138.940637,3.302779e+08,6563,0.791964,3.624580,0.218498,0.558981,0.835247,0.505268,0.567060,0.903908,416.229342,245,289,2.825168e+06,2.873337e+06,-4.816926e+04,0.031533,0.529412,3116,400.0,300.0,1480.0,0.716274,1505.710
1793,2026-04-02,9.127022e+07,7.316892e+07,1556,112244.092442,2.244882e+08,5386,0.801674,2.459599,0.325937,0.507319,0.559925,0.516632,0.553298,1.018357,468.306400,417,416,4.553407e+06,2.985051e+06,1.568355e+06,0.032706,0.300481,3138,410.0,300.0,1480.0,0.710102,1556.080
1794,2026-04-03,9.172124e+07,7.363709e+07,1566,46549.342765,9.309868e+07,3061,0.802836,1.015018,0.790957,0.437049,0.230733,0.475393,0.520670,1.087735,500.557848,83,84,4.974865e+06,4.455582e+06,5.192832e+05,0.048577,0.833333,3144,410.0,300.0,1480.0,0.706146,1564.165
1795,2026-04-04,9.187064e+07,7.364842e+07,1573,18230.108858,3.646022e+07,1791,0.801653,0.396865,2.019966,0.436346,0.090348,0.457393,0.499582,1.048236,482.498658,72,89,3.608810e+06,3.690071e+06,-8.126029e+04,0.040166,0.898876,3146,410.0,300.0,1480.0,0.706393,1567.570
1796,2026-04-05,9.265439e+07,7.401253e+07,1556,47141.227895,9.428245e+07,3354,0.798802,1.017571,0.785008,0.440054,0.232482,0.443678,0.488308,1.008237,466.466468,58,73,1.860587e+06,1.894873e+06,-3.428556e+04,0.020451,0.821918,3143,410.0,300.0,1480.0,0.705088,1555.010
1797,2026-04-06,9.362351e+07,7.460444e+07,1518,41223.922986,8.244784e+07,2248,0.796856,0.880632,0.904868,0.438128,0.201687,0.401720,0.488109,0.916902,425.530855,77,103,6.724085e+06,6.384565e+06,3.395205e+05,0.068194,0.563107,3133,400.0,300.0,1480.0,0.697811,1516.375


In [22]:
if plot_guard() and not lp_macro.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=lp_macro['date'], y=lp_macro['capital_utilization'], mode='lines', name='Capital utilization'))
    fig.add_trace(go.Scatter(x=lp_macro['date'], y=lp_macro['active_liquidity_depth'], mode='lines', name='Active liquidity depth'))
    fig.add_trace(go.Scatter(x=lp_macro['date'], y=lp_macro['volume_tvl_ratio'], mode='lines', name='Volume/TVL ratio'))
    fig.update_layout(title=f'Capital efficiency metrics - {POOL}', legend=dict(orientation='h'))
    fig.show()

In [23]:
if plot_guard() and not lp_macro.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=lp_macro['date'], y=lp_macro['apr_7d'], mode='lines', name='APR 7d'))
    fig.add_trace(go.Scatter(x=lp_macro['date'], y=lp_macro['apr_30d'], mode='lines', name='APR 30d'))
    fig.add_trace(go.Scatter(x=lp_macro['date'], y=lp_macro['apr7_vol_ratio'], mode='lines', name='APR7 / vol'))
    fig.add_trace(go.Scatter(x=lp_macro['date'], y=lp_macro['coupon_vol_ratio_500'], mode='lines', name='Coupon/vol 500'))
    fig.update_layout(title=f'Yield regime metrics - {POOL}', legend=dict(orientation='h'))
    fig.show()

In [24]:
if plot_guard() and not lp_macro.empty:
    fig = go.Figure()
    fig.add_trace(go.Bar(x=lp_macro['date'], y=lp_macro['mint_token0'], name='Mint flow'))
    fig.add_trace(go.Bar(x=lp_macro['date'], y=-lp_macro['burn_token0'], name='Burn flow'))
    fig.add_trace(go.Scatter(x=lp_macro['date'], y=lp_macro['net_lp_flow_token0'], mode='lines', name='Net LP flow', yaxis='y2'))
    fig.update_layout(
        title=f'LP flows - {POOL}',
        barmode='relative',
        yaxis=dict(title='Token0 flow'),
        yaxis2=dict(title='Net flow', overlaying='y', side='right'),
        legend=dict(orientation='h'),
    )
    fig.show()

In [25]:
if plot_guard() and not lp_macro.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=lp_macro['date'], y=lp_macro['open_range_count'], mode='lines', name='Open range count'))
    fig.add_trace(go.Scatter(x=lp_macro['date'], y=lp_macro['lp_concentration_top10_pct'], mode='lines', name='Top10 concentration'))
    fig.add_trace(go.Scatter(x=lp_macro['date'], y=lp_macro['range_overlap_density'], mode='lines', name='Overlap density'))
    fig.add_trace(go.Scatter(x=lp_macro['date'], y=lp_macro['range_width_median_ticks'], mode='lines', name='Median width'))
    fig.update_layout(title=f'Range structure metrics - {POOL}', legend=dict(orientation='h'))
    fig.show()

## 7. Intraday features

In [26]:
price_intraday = query_df(dedent(
    '''
    SELECT
        hour,
        open_price,
        high_price,
        low_price,
        close_price,
        volume_token0_equiv,
        swap_count,
        realized_vol_intraday
    FROM mart_price_intraday
    WHERE pool = %(pool)s
      AND date >= (SELECT max(date) - 14 FROM mart_price_intraday WHERE pool = %(pool)s)
    ORDER BY hour
    '''
), {'pool': POOL})
price_intraday.tail(20)

348 rows


,hour,open_price,high_price,low_price,close_price,volume_token0_equiv,swap_count,realized_vol_intraday
328,2026-04-05 16:00:00,2053.761396,2057.840433,2050.889933,2050.889968,2.773281e+06,118,0.004625
329,2026-04-05 17:00:00,2051.031918,2060.880221,2050.189576,2060.854815,4.117817e+06,165,0.004625
330,2026-04-05 18:00:00,2060.958286,2064.994992,2057.664545,2062.109709,2.187090e+06,90,0.004625
331,2026-04-05 19:00:00,2062.109872,2068.304356,2058.465721,2060.841143,2.812227e+06,89,0.004625
332,2026-04-05 20:00:00,2061.031648,2073.371776,2061.031648,2068.894535,3.165752e+06,121,0.004625
333,2026-04-05 21:00:00,2068.850731,2068.850731,2053.510016,2060.122880,5.948670e+06,132,0.004625
334,2026-04-05 22:00:00,2059.995764,2086.313562,2051.049139,2085.073893,1.459045e+07,262,0.004625
335,2026-04-05 23:00:00,2089.141590,2120.463204,2082.938443,2109.333463,1.796261e+07,445,0.004625
336,2026-04-06 00:00:00,2111.441018,2129.648903,2111.441018,2121.977379,1.443638e+07,265,0.003493
337,2026-04-06 01:00:00,2122.073931,2122.828079,2112.801756,2115.174936,7.489456e+06,186,0.003493


In [27]:
if plot_guard() and not price_intraday.empty:
    fig = go.Figure(data=[go.Candlestick(
        x=price_intraday['hour'],
        open=price_intraday['open_price'],
        high=price_intraday['high_price'],
        low=price_intraday['low_price'],
        close=price_intraday['close_price'],
        name='OHLC',
    )])
    fig.update_layout(title=f'Hourly OHLC - last ~2 weeks - {POOL}')
    fig.show()

In [28]:
if plot_guard() and not price_intraday.empty:
    fig = go.Figure()
    fig.add_trace(go.Bar(x=price_intraday['hour'], y=price_intraday['volume_token0_equiv'], name='Hourly volume'))
    fig.add_trace(go.Scatter(x=price_intraday['hour'], y=price_intraday['realized_vol_intraday'], mode='lines', name='Intraday vol', yaxis='y2'))
    fig.update_layout(
        title=f'Hourly volume and intraday vol - {POOL}',
        yaxis=dict(title='Volume'),
        yaxis2=dict(title='Intraday vol', overlaying='y', side='right'),
        legend=dict(orientation='h'),
    )
    fig.show()

## 8. Tick residency features

In [29]:
tick_daily = query_df(dedent(
    '''
    SELECT
        date,
        tick,
        time_seconds,
        volume_token0_equiv,
        swap_count
    FROM mart_tick_daily
    WHERE pool = %(pool)s
      AND date >= (SELECT max(date) - 30 FROM mart_tick_daily WHERE pool = %(pool)s)
    ORDER BY date, tick
    '''
), {'pool': POOL})
tick_daily.head(20)

11,499 rows


,date,tick,time_seconds,volume_usd,swap_count
0,2026-03-07,200340,36,107806.791442,8
1,2026-03-07,200342,12,77185.210730,2
2,2026-03-07,200343,12,96761.103083,1
3,2026-03-07,200344,36,324706.921555,6
4,2026-03-07,200345,48,455011.320816,4
5,2026-03-07,200346,36,56479.728376,4
6,2026-03-07,200348,36,189590.135466,8
7,2026-03-07,200349,12,13410.883204,6
8,2026-03-07,200350,0,217782.390690,2
9,2026-03-07,200351,60,7294.385614,6


In [30]:
if plot_guard() and not tick_daily.empty:
    heat = tick_daily.copy()
    heat['date_str'] = heat['date'].dt.strftime('%Y-%m-%d')
    fig = px.density_heatmap(
        heat,
        x='date_str',
        y='tick',
        z='time_seconds',
        histfunc='sum',
        title=f'Tick residency heatmap - last 30 days - {POOL}',
        labels={'date_str': 'Date', 'tick': 'Tick', 'time_seconds': 'Time seconds'},
    )
    fig.show()

In [31]:
if plot_guard() and not tick_daily.empty:
    fig = px.scatter(
        tick_daily,
        x='tick',
        y='volume_token0_equiv',
        color='swap_count',
        title=f'Tick-level volume vs tick - last 30 days - {POOL}',
        labels={'volume_token0_equiv': 'Volume USD'},
    )
    fig.show()

## 9. Open positions features

In [32]:
open_positions = query_df(dedent(
    '''
    SELECT
        owner,
        tick_lower,
        tick_upper,
        (tick_upper - tick_lower) AS width_ticks,
        net_liquidity,
        is_active,
        position_amount0,
        position_amount1,
        position_token0,
        current_tick,
        price_token1_in_token0
    FROM mart_open_positions
    WHERE pool = %(pool)s
    ORDER BY position_token0 DESC
    LIMIT 5000
    '''
), {'pool': POOL})
open_positions.head(20)

3,145 rows


,owner,tick_lower,tick_upper,width_ticks,net_liquidity,is_active,position_amount0,position_amount1,position_token0,current_tick,price_token1_in_token0
0,0xc36442b4a4522e871399cd717abdd847ab11fe88,199380,200860,1480,7598705807111945268,1,2.171811e+07,1691.572688,2.535515e+07,199587,2150.092004
1,0xc36442b4a4522e871399cd717abdd847ab11fe88,200150,201940,1790,2794337315720186614,0,1.078466e+07,0.000000,1.078466e+07,199587,2150.092004
2,0xc36442b4a4522e871399cd717abdd847ab11fe88,195610,209470,13860,350427488554237191,1,6.335152e+06,1362.898631,9.265510e+06,199587,2150.092004
3,0xc36442b4a4522e871399cd717abdd847ab11fe88,194710,212350,17640,171327716065031450,1,3.747333e+06,799.592201,5.466530e+06,199587,2150.092004
4,0xc36442b4a4522e871399cd717abdd847ab11fe88,199310,200870,1560,1352857039838262121,1,3.896068e+06,402.043602,4.760499e+06,199587,2150.092004
5,0xc36442b4a4522e871399cd717abdd847ab11fe88,198320,202640,4320,323586592167242883,1,2.123733e+06,428.528893,3.045110e+06,199587,2150.092004
6,0xc36442b4a4522e871399cd717abdd847ab11fe88,198910,200310,1400,847800319555502710,1,1.394664e+06,608.985969,2.704040e+06,199587,2150.092004
7,0xc36442b4a4522e871399cd717abdd847ab11fe88,-887270,887270,1774540,17878839871356764,1,8.290254e+05,385.576727,1.658051e+06,199587,2150.092004
8,0xc36442b4a4522e871399cd717abdd847ab11fe88,196950,206450,9500,65989735192964238,1,8.887041e+05,175.821962,1.266737e+06,199587,2150.092004
9,0xc36442b4a4522e871399cd717abdd847ab11fe88,198490,200310,1820,248788741341878692,1,4.092671e+05,286.487699,1.025242e+06,199587,2150.092004


In [33]:
if plot_guard() and not open_positions.empty:
    fig = px.histogram(
        open_positions,
        x='width_ticks',
        nbins=60,
        color=open_positions['is_active'].astype(str),
        title=f'Open position width distribution - {POOL}',
        labels={'width_ticks': 'Width (ticks)', 'color': 'is_active'},
    )
    fig.show()

In [34]:
if plot_guard() and not open_positions.empty:
    fig = px.scatter(
        open_positions,
        x='width_ticks',
        y='position_token0',
        color=open_positions['is_active'].astype(str),
        hover_data=['owner', 'tick_lower', 'tick_upper'],
        title=f'Position size vs width - {POOL}',
        labels={'position_token0': 'Position token0 equiv', 'color': 'is_active'},
        log_y=True,
    )
    fig.show()

In [35]:
if plot_guard() and not open_positions.empty:
    top_positions = open_positions.nlargest(20, 'position_token0').copy()
    top_positions['owner_short'] = top_positions['owner'].str.slice(0, 10)
    fig = px.bar(
        top_positions.sort_values('position_token0'),
        x='position_token0',
        y='owner_short',
        color=top_positions['is_active'].astype(str),
        orientation='h',
        title=f'Top open positions by size - {POOL}',
        labels={'position_token0': 'Position token0 equiv', 'owner_short': 'Owner'},
    )
    fig.show()

## 10. Quick summary table

In [36]:
summary = pd.DataFrame([
    {
        'pool': POOL,
        'latest_daily_date': tvl_daily['date'].max() if not tvl_daily.empty else None,
        'snapshot_target_block': int(tvl_snapshot.iloc[0]['target_block']) if not tvl_snapshot.empty and pd.notna(tvl_snapshot.iloc[0]['target_block']) else None,
        'total_tvl_current': float(tvl_snapshot.iloc[0]['total_tvl_token0']) if not tvl_snapshot.empty else None,
        'active_tvl_current': float(tvl_snapshot.iloc[0]['active_tvl_token0']) if not tvl_snapshot.empty else None,
        'out_of_range_tvl_current': float(tvl_snapshot.iloc[0]['out_of_range_tvl_token0']) if not tvl_snapshot.empty else None,
        'current_bin_tvl': float(tvl_snapshot.iloc[0]['current_bin_tvl_token0']) if not tvl_snapshot.empty else None,
        'latest_daily_volume': float(volume_fee_daily['volume_token0_equiv'].dropna().iloc[-1]) if not volume_fee_daily.empty else None,
        'latest_daily_fees': float(volume_fee_daily['fees_token0_equiv'].dropna().iloc[-1]) if not volume_fee_daily.empty else None,
        'latest_apr_7d': float(lp_macro['apr_7d'].dropna().iloc[-1]) if not lp_macro.empty and lp_macro['apr_7d'].dropna().shape[0] else None,
        'latest_realized_vol_7d': float(price_analytics['realized_vol_7d'].dropna().iloc[-1]) if not price_analytics.empty and price_analytics['realized_vol_7d'].dropna().shape[0] else None,
        'distribution_bins': int(len(tvl_distribution)),
        'open_positions_sampled': int(len(open_positions)),
    }
])
summary

,pool,latest_daily_date,snapshot_target_block,total_tvl_current,active_tvl_current,out_of_range_tvl_current,current_bin_tvl,latest_daily_volume,latest_daily_fees,latest_apr_7d,latest_realized_vol_7d,distribution_bins,open_positions_sampled
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-06,24820333,9.362351e+07,7.460444e+07,1.901907e+07,336558.14942,8.244784e+07,41223.922986,0.40172,0.438128,1537,3145
